# Automated Eval Pipelines

Deep Dive 02. The [parent notebook](/courses/llm-eng/04-eval-concepts.html) introduced the three-tier evaluation stack and showed that RAGAS scores drop when context is degraded and that an LLM judge calibrates correctly across good, partial, and hallucinated answers. This notebook operationalizes those ideas into a production eval pipeline: we build a `DatasetCurator` that generates a 20-example golden dataset from the SEC filings corpus using synthetic question-answer generation, implement an `LLMJudge` with three judge prompts (relevance, faithfulness, and pairwise preference), assemble a `CIEvalHarness` that computes RAGAS metrics, stores results as JSON, and compares against a baseline to detect regressions, introduce a `PromptVersionRegistry` for tracking metric snapshots per prompt hash, and close with a `bootstrap_ci` function for principled uncertainty quantification on eval scores.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Dataset Curation Strategy

A **golden dataset** is a fixed collection of (question, reference answer, context) triples whose quality you trust enough to use as a regression benchmark. "Fixed" is the key word: once you adopt a golden set, you do not retroactively edit it to match model outputs — that defeats the purpose. The dataset defines what the system should do, not what it currently does.

<br>

**Three sourcing strategies.** In practice, golden datasets are assembled from multiple sources: (1) **production logs** — sampled queries from real users, annotated by domain experts; (2) **analyst-curated edge cases** — questions that the team knows are tricky, such as queries requiring exact identifier matching ("What is the FINRA Rule 4110 net capital requirement?") or multi-hop reasoning across sections; and (3) **synthetic generation** — an LLM reads a passage and writes a question whose answer is entirely contained in that passage. Synthetic generation is the fastest way to bootstrap a dataset when production logs are unavailable.

<br>

**Synthetic generation quality.** The generator LLM tends to ask simple extractive questions — "What was the CET1 ratio?" — rather than multi-step reasoning questions. This is fine for catching hallucinations and faithfulness regressions, which is the primary use of automated evals. For evaluating reasoning quality, supplement with analyst-curated cases. We implement a `DatasetCurator` that wraps the synthetic generation pattern with Pydantic output parsing so each generated example is structurally valid.

:::{.callout-note}
Synthetic evals are a starting point, not a destination. Treat them as a fast regression floor. As your system matures, replace synthetic examples with real annotated queries from production traffic — those reflect the actual distribution of questions your users ask.

:::

We use the same 20-passage SEC filings corpus from [notebook 07](/courses/llm-eng/07-rag-pipeline.html):

In [ ]:
CORPUS = [
    # Section: risk_factors (indices 0-4)
    {"text": "Interest rate risk represents one of the most significant market risks facing the firm. A 100 basis point increase in interest rates would reduce the fair value of our fixed-rate debt portfolio by approximately $2.3 billion.", "section": "risk_factors"},
    {"text": "Credit risk arises from the potential that a counterparty will fail to perform its obligations. We manage credit risk through diversification, collateral requirements, and credit limits by counterparty.", "section": "risk_factors"},
    {"text": "Operational risk includes the risk of loss resulting from inadequate or failed internal processes, people, systems, or external events, including cybersecurity threats and technology failures.", "section": "risk_factors"},
    {"text": "Our derivatives portfolio had a notional value of $1.2 trillion at year-end. Net market value exposure after netting and collateral was $18.4 billion, primarily concentrated in interest rate and foreign exchange derivatives.", "section": "risk_factors"},
    {"text": "Our Value at Risk (VaR) at the 99th percentile for a one-day holding period was $142 million, reflecting the diversified nature of our trading portfolios across equities, fixed income, and commodities.", "section": "risk_factors"},

    # Section: mda (indices 5-9)
    {"text": "Net revenues for the fiscal year were $47.4 billion, an increase of 8% compared to the prior year. The increase was driven primarily by higher net interest income reflecting the rising interest rate environment.", "section": "mda"},
    {"text": "Investment banking revenues decreased 23% to $6.1 billion, reflecting lower advisory fees amid reduced M&A activity and a challenging environment for equity and debt underwriting.", "section": "mda"},
    {"text": "Return on equity for the year was 12.4%, compared to 15.1% in the prior year. Book value per share increased to $312.50, up from $290.20.", "section": "mda"},
    {"text": "Net interest margin expanded 18 basis points to 2.94%, driven by higher short-term rates partially offset by increased funding costs. Provision for credit losses increased to $2.1 billion, reflecting normalization from historically low levels.", "section": "mda"},
    {"text": "Assets under management in our investment management segment grew to $2.8 trillion, an increase of 6% from prior year. Prime brokerage revenues increased 12% to $4.3 billion on higher client balances and margin loan activity.", "section": "mda"},

    # Section: capital_liquidity (indices 10-14)
    {"text": "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and our internal target of 13%.", "section": "capital_liquidity"},
    {"text": "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the regulatory requirement of 100%. Our high-quality liquid assets totaled $280 billion at year-end.", "section": "capital_liquidity"},
    {"text": "Under Basel III framework requirements, our leverage ratio was 5.8%, comfortably above the 3% minimum. Our total risk-weighted assets were $1.47 trillion at year-end, consistent with prior year.", "section": "capital_liquidity"},
    {"text": "We are subject to FINRA Rule 4110 and SEC Rule 15c3-1 (the Net Capital Rule), which require us to maintain minimum net capital of not less than the greater of $250,000 or 2% of aggregate debit items. Our net capital exceeded the minimum by $12.4 billion.", "section": "capital_liquidity"},
    {"text": "The liquidity stress test results indicate the firm could withstand a 30-day severe market stress scenario while maintaining positive liquidity. The Internal Liquidity Adequacy Assessment Process (ILAAP) was completed in Q3 and reviewed by the Board Risk Committee.", "section": "capital_liquidity"},

    # Section: guidance (indices 15-19)
    {"text": "Looking ahead to fiscal 2025, management expects continued revenue growth in the range of 4-6%, supported by a stable rate environment and improving capital markets activity.", "section": "guidance"},
    {"text": "We plan to return $8 billion to shareholders through dividends and share repurchases in fiscal 2025, subject to regulatory approval and market conditions.", "section": "guidance"},
    {"text": "Earnings per share for fiscal 2024 were $42.30, compared to $47.20 in the prior year, reflecting lower net income partially offset by the reduction in diluted share count from ongoing repurchases.", "section": "guidance"},
    {"text": "Management has identified three strategic priorities for fiscal 2025: (1) expanding the wealth management client base to $500 billion in AUM, (2) growing transaction banking revenues by 15%, and (3) reducing the expense ratio below 65%.", "section": "guidance"},
    {"text": "We expect our CET1 ratio to remain in the 13.5-15.0% range through fiscal 2025, subject to regulatory stress test outcomes. Any excess capital above 14.5% will be returned to shareholders under our capital return policy.", "section": "guidance"},
]

print(f"Corpus: {len(CORPUS)} passages across {len({c['section'] for c in CORPUS})} sections")

We define `DatasetCurator` with a Pydantic output schema for the generated examples:

In [ ]:
class GeneratedExample(BaseModel):
    question: str
    reference_answer: str


class DatasetCurator:
    """Generate (question, reference_answer, context) triples from a corpus."""

    _SYSTEM = (
        "You are a financial analyst building an evaluation dataset. "
        "Given a passage from a SEC filing, write one factual question whose complete answer "
        "is contained verbatim or near-verbatim in the passage. "
        "The reference_answer must be a single sentence using only information in the passage."
    )

    def __init__(self, generator: LLMClient):
        self._llm = generator  # <1>

    def generate_one(self, passage: dict) -> dict:
        """Return a single eval example for one corpus passage."""
        messages = [
            {"role": "system", "content": self._SYSTEM},
            {"role": "user", "content": f"Passage: {passage['text']}"},
        ]
        result: GeneratedExample = self._llm.complete(
            messages, response_format=GeneratedExample  # <2>
        )
        return {
            "question": result.question,
            "reference_answer": result.reference_answer,
            "context": passage["text"],
            "section": passage["section"],
        }

    def generate_dataset(self, corpus: list[dict]) -> list[dict]:  # <3>
        """Generate one example per passage."""
        return [self.generate_one(p) for p in corpus]

1. We inject `generator` so the curator can be tested with a mock client without making real API calls.
2. Structured output via `response_format=GeneratedExample` guarantees we always receive a valid `question` and `reference_answer` string — no regex parsing needed and no risk of a malformed JSON response crashing the curator.
3. `generate_dataset` runs one LLM call per passage. For a 20-passage corpus at `gpt-4o-mini` pricing, this costs roughly $0.003 — cheap enough to regenerate the dataset on demand when the corpus changes.

Generating the 20-example golden dataset:

In [ ]:
curator = DatasetCurator(generator=LLMClient(model="gpt-4o-mini"))
golden_dataset = curator.generate_dataset(CORPUS)

print(f"Generated {len(golden_dataset)} examples")
for ex in golden_dataset[:3]:
    print(f"  [{ex['section']}] Q: {ex['question'][:70]}")
    print(f"           A: {ex['reference_answer'][:70]}")

We persist the golden dataset to disk so it can be version-controlled and loaded without re-running the generator:

In [ ]:
import pathlib

GOLDEN_PATH = pathlib.Path("/tmp/golden_dataset.json")
GOLDEN_PATH.write_text(json.dumps(golden_dataset, indent=2))

# Reload to confirm round-trip
loaded = json.loads(GOLDEN_PATH.read_text())
print(f"Saved and reloaded {len(loaded)} examples from {GOLDEN_PATH}")

## LLM-as-Judge Patterns

The [parent notebook](/courses/llm-eng/04-eval-concepts.html) implemented a single G-Eval scorer on a 1–5 rubric. We now implement three specialized judge prompts, each targeting a distinct failure mode.

**Relevance judge.** Scores 0–3 on whether the answer addresses the question. A score of 0 means the answer is completely off-topic; 3 means it directly answers the question with the appropriate level of detail. This catches cases where the LLM drifts toward a related but different question — common when the context contains multiple relevant passages.

<br>

**Faithfulness judge.** Binary (0 or 1): does the answer contain only claims that are supported by the retrieved context? A 0 does not mean the answer is factually wrong in the real world — it means the answer makes claims beyond what the context warrants. This distinction matters in compliance contexts: an answer that adds a true fact not present in the retrieved passage is still a faithfulness failure, because the model is drawing on parametric knowledge rather than the authorized source.

<br>

**Pairwise preference judge.** Given two candidate answers to the same question, which is better? Pairwise comparisons are more reliable than absolute scores because humans and models are better at relative ranking than absolute quality judgments. This pattern is useful for A/B testing prompt versions: instead of comparing aggregate scores (which can obscure which individual cases changed), we run head-to-head comparisons on each test case.

All three judges include chain-of-thought extraction: the reasoning field is returned alongside the score. Logging the reasoning is essential for debugging — when a regression is detected, the reasoning explains which specific claim the judge flagged.

We define Pydantic output schemas for the three judge types:

In [ ]:
class RelevanceResult(BaseModel):
    reasoning: str
    score: int  # 0-3


class FaithfulnessResult(BaseModel):
    reasoning: str
    is_faithful: bool  # True if all claims are supported by context
    unsupported_claims: list[str]  # empty list if faithful


class PreferenceResult(BaseModel):
    reasoning: str
    preferred: str  # "A" | "B" | "tie"
    confidence: str  # "clear" | "slight" | "tie"

We implement `LLMJudge` with one method per judge type:

In [ ]:
class LLMJudge:
    """Three-prompt LLM judge for RAG answer evaluation."""

    _RELEVANCE_SYSTEM = """
You are a financial QA evaluator. Score the answer on relevance to the question (0-3):
3 = Directly and completely answers the question with appropriate detail.
2 = Answers the question but omits important details or is overly verbose.
1 = Partially answers the question or addresses only part of it.
0 = Does not answer the question; off-topic or irrelevant.
First reason step by step, then give a score.
"""

    _FAITHFULNESS_SYSTEM = """
You are a financial compliance evaluator checking answer grounding.
Determine whether every claim in the answer is directly supported by the provided context.
An answer is NOT faithful if it introduces any specific figure, fact, or claim not present in the context,
even if that claim might be true in the real world.
List any unsupported claims. If none, return an empty list.
"""

    _PREFERENCE_SYSTEM = """
You are a financial QA quality evaluator. Given a question and two candidate answers (A and B),
determine which answer is better for a financial analyst audience.
Consider: factual accuracy, completeness, conciseness, and appropriate use of specific figures.
First reason through the comparison, then state your preference: "A", "B", or "tie".
Also state your confidence: "clear" if one answer is obviously better, "slight" for a marginal difference, "tie" if equivalent.
"""

    def __init__(self, judge_model: str = "gpt-4o"):  # <1>
        self._llm = LLMClient(model=judge_model, temperature=0.0)

    def relevance(
        self, question: str, answer: str
    ) -> RelevanceResult:
        """Score how well the answer addresses the question (0-3)."""
        messages = [
            {"role": "system", "content": self._RELEVANCE_SYSTEM.strip()},
            {"role": "user", "content": f"Question: {question}\n\nAnswer: {answer}"},
        ]
        return self._llm.complete(messages, response_format=RelevanceResult)

    def faithfulness(
        self, question: str, answer: str, context: str
    ) -> FaithfulnessResult:
        """Check whether every claim in the answer is grounded in the context."""
        messages = [
            {"role": "system", "content": self._FAITHFULNESS_SYSTEM.strip()},
            {
                "role": "user",
                "content": (
                    f"Context: {context}\n\n"
                    f"Question: {question}\n\n"
                    f"Answer: {answer}"
                ),
            },
        ]
        return self._llm.complete(messages, response_format=FaithfulnessResult)  # <2>

    def pairwise(
        self, question: str, answer_a: str, answer_b: str, context: str = ""
    ) -> PreferenceResult:
        """Compare two candidate answers and return a preference."""
        context_block = f"Context: {context}\n\n" if context else ""
        messages = [
            {"role": "system", "content": self._PREFERENCE_SYSTEM.strip()},
            {
                "role": "user",
                "content": (
                    f"{context_block}"
                    f"Question: {question}\n\n"
                    f"Answer A: {answer_a}\n\n"
                    f"Answer B: {answer_b}"
                ),
            },
        ]
        return self._llm.complete(messages, response_format=PreferenceResult)  # <3>

1. We always use a stronger model (`gpt-4o`) as the judge than the model under test (`gpt-4o-mini`). A model judging its own outputs introduces self-serving bias.
2. The `faithfulness` judge returns `unsupported_claims` as a list rather than a count. This is more actionable for debugging: when a regression is detected you can see exactly which claims the judge flagged rather than just that the binary score flipped.
3. `pairwise` is deliberately position-agnostic: the `preferred` field is always "A" or "B" relative to what was passed in, not an absolute label. In practice, run each comparison twice with A and B swapped and take the majority verdict to cancel out position bias — the judge tends to slightly favor whichever answer appears first.

We exercise all three judges on hand-crafted examples to verify calibration:

In [ ]:
judge = LLMJudge(judge_model="gpt-4o")

q = "What was the CET1 capital ratio at year-end?"
ctx = CORPUS[10]["text"]

good_answer     = "The CET1 ratio was 14.8% at year-end, well above the 4.5% regulatory minimum."
vague_answer    = "The CET1 ratio was well above the regulatory minimum."
halluc_answer   = "The CET1 ratio was 11.2%, just below the 12% regulatory requirement."

# Relevance judge
print("=== Relevance ===")
for label, ans in [("good", good_answer), ("vague", vague_answer), ("hallucinated", halluc_answer)]:
    r = judge.relevance(q, ans)
    print(f"  [{label}] score={r.score}/3  reasoning: {r.reasoning[:80]}...")

# Faithfulness judge
print("\n=== Faithfulness ===")
for label, ans in [("good", good_answer), ("hallucinated", halluc_answer)]:
    r = judge.faithfulness(q, ans, ctx)
    print(f"  [{label}] faithful={r.is_faithful}  unsupported={r.unsupported_claims}")

# Pairwise judge
print("\n=== Pairwise ===")
r = judge.pairwise(q, good_answer, halluc_answer, context=ctx)
print(f"  preferred={r.preferred}  confidence={r.confidence}")
print(f"  reasoning: {r.reasoning[:120]}...")

The relevance judge correctly distinguishes between the specific answer (score 3), the vague answer (score 2, missing the exact figure), and the hallucinated answer (score 0 or 1, answering a different question with fabricated numbers). The faithfulness judge returns `is_faithful=False` for the hallucinated answer and lists the fabricated figures in `unsupported_claims`. The pairwise judge prefers the good answer with clear confidence.

## Eval Harness with CI Integration

The `CIEvalHarness` has four responsibilities: (1) run a pipeline over all golden dataset examples and collect answers; (2) compute RAGAS metrics (faithfulness, answer relevancy, context recall) using `ragas.evaluate`; (3) persist results as a JSON snapshot keyed by a run identifier; and (4) compare the current snapshot against a baseline and return exit code 0 (pass) or 1 (fail) based on configurable thresholds.

The exit code convention is what allows the harness to integrate with GitHub Actions. The CI workflow runs the harness as a Python script step; if the harness returns exit code 1, the step fails and the pull request is blocked. No custom GitHub Action plugin is needed — it is a plain `python eval_harness.py` call in the workflow YAML.

:::{.callout-important}
The baseline JSON must be committed to the repository alongside the code. A new baseline is only committed when the team consciously decides to accept a new performance level — not automatically on every merge. Treat baseline updates like schema migrations: deliberate, reviewed, and documented.

:::

We define a simple pipeline function that generates answers from context using `llm`:

In [ ]:
def make_pipeline(system_prompt: str):
    """Return a (question, context) -> answer function backed by the given system prompt."""
    def pipeline(question: str, context: str) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ]
        return llm.complete(messages)
    return pipeline


STRONG_PROMPT = (
    "You are a financial analyst assistant. "
    "Answer using ONLY the provided context. "
    "Include specific numbers and figures. "
    "If the context does not contain the answer, say so explicitly."
)

print("Pipeline defined.")

We now implement `CIEvalHarness`:

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness as ragas_faithfulness
from ragas.metrics import answer_relevancy, context_recall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pathlib, datetime


class CIEvalHarness:
    """RAGAS-based eval harness with baseline comparison and JSON persistence."""

    THRESHOLDS = {  # <1>
        "faithfulness": 0.75,
        "answer_relevancy": 0.70,
        "context_recall": 0.70,
    }

    def __init__(
        self,
        pipeline_fn,
        golden_dataset: list[dict],
        results_dir: str = "/tmp/eval_results",
    ):
        self._pipeline = pipeline_fn
        self._dataset = golden_dataset
        self._dir = pathlib.Path(results_dir)
        self._dir.mkdir(parents=True, exist_ok=True)
        self._judge_llm = ChatOpenAI(model="gpt-4o-mini")
        self._embed = OpenAIEmbeddings(model="text-embedding-3-small")

    def _build_ragas_dataset(self) -> Dataset:  # <2>
        """Run the pipeline over all golden examples to produce a RAGAS Dataset."""
        rows = []
        for ex in self._dataset:
            answer = self._pipeline(ex["question"], ex["context"])
            rows.append({
                "question": ex["question"],
                "answer": answer,
                "contexts": [ex["context"]],
                "ground_truth": ex["reference_answer"],
            })
        return Dataset.from_list(rows)

    def run(self, run_id: str | None = None) -> dict:
        """Run the full eval suite and persist results. Returns the metrics dict."""
        run_id = run_id or datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
        ds = self._build_ragas_dataset()
        result = evaluate(
            dataset=ds,
            metrics=[ragas_faithfulness, answer_relevancy, context_recall],
            llm=self._judge_llm,
            embeddings=self._embed,
        )
        metrics = {
            "run_id": run_id,
            "faithfulness": float(result["faithfulness"]),
            "answer_relevancy": float(result["answer_relevancy"]),
            "context_recall": float(result["context_recall"]),
        }
        out_path = self._dir / f"{run_id}.json"
        out_path.write_text(json.dumps(metrics, indent=2))  # <3>
        return metrics

    def compare_to_baseline(
        self, current: dict, baseline_path: str, delta_threshold: float = 0.05
    ) -> tuple[int, list[str]]:
        """Compare current metrics to a baseline file. Returns (exit_code, failure_messages)."""
        baseline = json.loads(pathlib.Path(baseline_path).read_text())
        failures = []
        metric_keys = ["faithfulness", "answer_relevancy", "context_recall"]
        for key in metric_keys:
            curr_val = current.get(key, 0.0)
            base_val = baseline.get(key, 0.0)
            drop = base_val - curr_val
            if drop > delta_threshold:  # <4>
                failures.append(
                    f"{key}: dropped {drop:.3f} (baseline={base_val:.3f}, current={curr_val:.3f})"
                )
            floor = self.THRESHOLDS.get(key, 0.0)
            if curr_val < floor:
                failures.append(
                    f"{key}: below floor threshold ({curr_val:.3f} < {floor:.3f})"
                )
        exit_code = 1 if failures else 0
        return exit_code, failures

1. `THRESHOLDS` defines absolute floors for each metric. Even if the current run shows no regression relative to the baseline, a score below the floor indicates the pipeline is not production-ready and the CI step fails.
2. `_build_ragas_dataset` calls the pipeline once per golden example. For a 20-example dataset this takes roughly 30 seconds end-to-end including the RAGAS metric computation; budget ~60 seconds for the full CI step.
3. Each run is saved as an individual JSON file named by the run ID. This creates an immutable audit trail of every eval run — useful for tracking score trends over time and for post-incident analysis.
4. `delta_threshold=0.05` means a 5-point drop triggers a failure. This is deliberately loose enough to absorb LLM stochasticity (which can cause ±2–3 point swings between identical runs) while still catching real regressions.

Running the harness on the strong pipeline and saving the result as our baseline:

In [ ]:
harness = CIEvalHarness(
    pipeline_fn=make_pipeline(STRONG_PROMPT),
    golden_dataset=golden_dataset,
    results_dir="/tmp/eval_results",
)

baseline_metrics = harness.run(run_id="baseline-v1")
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    if k != "run_id":
        print(f"  {k}: {v:.3f}")

# Save as the canonical baseline
import shutil
shutil.copy("/tmp/eval_results/baseline-v1.json", "/tmp/eval_results/baseline.json")
print("\nBaseline saved to /tmp/eval_results/baseline.json")

The GitHub Actions integration is a single workflow step. No custom action is required:

In [ ]:
#| echo: false
github_actions_yaml = """
name: Eval Regression Check

on:
  pull_request:
    paths:
      - 'src/prompts/**'
      - 'src/pipeline/**'

jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.13'
      - run: pip install -r requirements.txt
      - name: Run eval harness
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: |
          python scripts/run_eval.py \\
            --golden evals/golden_dataset.json \\
            --baseline evals/baseline.json \\
            --delta-threshold 0.05
""".strip()

print(github_actions_yaml)

The `paths` trigger means the eval workflow only runs when prompt or pipeline files change — not on documentation changes or test file changes. This keeps CI costs low: at $0.003 per 20-example run, running on every PR touching `src/prompts/` costs approximately $0.50 per month at typical PR velocity.

## Prompt Regression Detection

We now walk through a concrete regression scenario: the system prompt is changed, we run the harness, and we confirm that the harness catches the faithfulness drop. We also introduce a `PromptVersionRegistry` that maps prompt hashes to their metric snapshots — this makes it possible to answer "when did faithfulness first drop below 0.80, and which commit caused it?" without reconstructing the history from commit logs.

We define `PromptVersionRegistry`:

In [ ]:
import hashlib


class PromptVersionRegistry:
    """Map prompt hash -> metric snapshot for regression traceability."""

    def __init__(self, registry_path: str = "/tmp/prompt_registry.json"):
        self._path = pathlib.Path(registry_path)
        self._data: dict = json.loads(self._path.read_text()) if self._path.exists() else {}

    @staticmethod
    def hash_prompt(prompt: str) -> str:  # <1>
        """Return the first 12 hex characters of SHA-256(prompt)."""
        return hashlib.sha256(prompt.encode()).hexdigest()[:12]

    def register(self, prompt: str, metrics: dict, label: str = "") -> str:
        """Store metrics under the prompt's hash. Returns the hash."""
        h = self.hash_prompt(prompt)
        self._data[h] = {
            "label": label,
            "prompt_preview": prompt[:80],
            "metrics": metrics,
            "registered_at": datetime.datetime.utcnow().isoformat(),
        }
        self._path.write_text(json.dumps(self._data, indent=2))  # <2>
        return h

    def lookup(self, prompt: str) -> dict | None:
        """Return the stored metrics for a given prompt, or None."""
        return self._data.get(self.hash_prompt(prompt))

    def history(self) -> list[dict]:  # <3>
        """Return all registered entries sorted by registration time."""
        return sorted(self._data.values(), key=lambda x: x["registered_at"])

1. SHA-256 hash of the full prompt text gives a stable, content-addressed identifier. Two prompts that differ by a single space will have completely different hashes — this is intentional, since even minor wording changes can affect model behavior.
2. The registry persists to disk on every `register` call. In production, use a proper key-value store (Redis, DynamoDB) and append rather than overwrite, so concurrent eval runs do not race.
3. `history()` exposes the full audit trail. You can iterate over it to plot faithfulness scores over time or to find the first commit at which a metric dropped below a threshold.

We register the baseline prompt, then run the harness with a degraded prompt and demonstrate that the harness catches the regression:

In [ ]:
WEAK_PROMPT = "Answer the question based on the context."  # deliberately vague

registry = PromptVersionRegistry()

# Register the strong-prompt baseline
h_strong = registry.register(STRONG_PROMPT, baseline_metrics, label="strong-v1")
print(f"Registered strong prompt: hash={h_strong}")

# Run the harness with the weak prompt
harness_weak = CIEvalHarness(
    pipeline_fn=make_pipeline(WEAK_PROMPT),
    golden_dataset=golden_dataset,
    results_dir="/tmp/eval_results",
)
weak_metrics = harness_weak.run(run_id="weak-prompt-run")

# Register weak prompt metrics
h_weak = registry.register(WEAK_PROMPT, weak_metrics, label="weak-v1")
print(f"Registered weak prompt:   hash={h_weak}")

print("\n--- Metric comparison ---")
for k in ["faithfulness", "answer_relevancy", "context_recall"]:
    delta = weak_metrics[k] - baseline_metrics[k]
    flag = " ← REGRESSION" if delta < -0.05 else ""
    print(f"  {k:20s}: baseline={baseline_metrics[k]:.3f}  current={weak_metrics[k]:.3f}  Δ={delta:+.3f}{flag}")

We now invoke `compare_to_baseline` to get the formal CI verdict:

In [ ]:
exit_code, failures = harness_weak.compare_to_baseline(
    current=weak_metrics,
    baseline_path="/tmp/eval_results/baseline.json",
    delta_threshold=0.05,
)

print(f"CI exit code: {exit_code} ({'PASS' if exit_code == 0 else 'FAIL'})")
if failures:
    print("Failures detected:")
    for f in failures:
        print(f"  - {f}")
else:
    print("No regressions detected.")

The harness returns exit code 1 and lists the specific metric(s) that regressed, giving a clear signal to the PR author about what to fix. We can also inspect the registry history to confirm the audit trail is intact:

In [ ]:
print("Prompt version history:")
for entry in registry.history():
    m = entry["metrics"]
    faith = m.get("faithfulness", float("nan"))
    print(f"  [{entry['label']:12s}] faithfulness={faith:.3f}  preview: {entry['prompt_preview'][:60]}...")

## Statistical Significance

Saying "our faithfulness score went from 0.80 to 0.82" is only meaningful if the difference is larger than the noise inherent in the evaluation process. There are two sources of variance in an eval run: (1) **LLM stochasticity** — even at temperature 0.0 the model can return slightly different outputs due to floating-point nondeterminism; and (2) **dataset sampling variance** — if we ran the eval on a different 20-example sample from the same distribution, we would get a different score.

<br>

The **bootstrap confidence interval** addresses source (2): we treat the observed per-example scores as a sample from the true score distribution and simulate the sampling variance by drawing many bootstrap resamples. Formally, given $n$ per-example scores $s_1, \ldots, s_n$, the bootstrap procedure draws $B$ samples of size $n$ with replacement, computes the mean $\bar{s}^{(b)}$ for each, and reports the $\alpha/2$ and $1 - \alpha/2$ quantiles of the resulting distribution as the confidence interval:

$$\text{CI}_{1-\alpha} = \left[\hat{q}_{\alpha/2}, \hat{q}_{1-\alpha/2}\right]$$

where $\hat{q}_p$ is the $p$-th quantile of $\{\bar{s}^{(1)}, \ldots, \bar{s}^{(B)}\}$. For small eval sets ($n < 50$), the $95\%$ confidence interval on a mean score is typically $\pm 0.05$ to $\pm 0.10$ — meaning a change of $0.02$ is not statistically distinguishable from noise.

:::{.callout-caution}
Bootstrap CIs address sampling variance within a fixed eval set. They do not address distribution shift between the eval set and production traffic. A tight CI with a biased eval set is false precision — it confirms that you are consistently measuring the wrong thing.

:::

We implement `bootstrap_ci` and apply it to the per-example faithfulness scores from both pipeline runs:

In [ ]:
def bootstrap_ci(
    scores: list[float],
    n_bootstrap: int = 1000,
    confidence: float = 0.95,
    seed: int = 42,
) -> tuple[float, float, float]:
    """Bootstrap confidence interval for the mean of `scores`.

    Returns (mean, lower_bound, upper_bound).
    """
    rng = np.random.default_rng(seed)
    arr = np.array(scores, dtype=float)
    n = len(arr)
    boot_means = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        sample = rng.choice(arr, size=n, replace=True)  # <1>
        boot_means[b] = sample.mean()
    alpha = 1.0 - confidence
    lo = float(np.quantile(boot_means, alpha / 2))       # <2>
    hi = float(np.quantile(boot_means, 1.0 - alpha / 2))
    return float(arr.mean()), lo, hi

1. Drawing `size=n` with replacement is the definition of a bootstrap resample. Each resample will over-represent some observations and exclude others, mimicking the variance you would observe if you had drawn a different 20-example sample from the same distribution.
2. The percentile method confidence interval uses the empirical quantiles of the bootstrap distribution directly. This is the simplest correct approach; more refined methods (BCa — bias-corrected accelerated) are warranted when the distribution is heavily skewed, but for eval means on 20+ examples the percentile method is adequate.

We simulate per-example faithfulness scores for the two pipeline runs and compute confidence intervals:

In [ ]:
# Simulate per-example binary faithfulness scores (0 or 1)
# In a real run these come from RAGAS per-row scores via result.to_pandas()
rng = np.random.default_rng(0)

# Strong pipeline: faithfulness ~ 0.85
strong_scores = rng.binomial(1, 0.85, size=20).tolist()

# Weak pipeline: faithfulness ~ 0.71 (simulating the regression)
weak_scores = rng.binomial(1, 0.71, size=20).tolist()

strong_mean, strong_lo, strong_hi = bootstrap_ci(strong_scores)
weak_mean,   weak_lo,   weak_hi   = bootstrap_ci(weak_scores)

print("95% bootstrap CIs on faithfulness:")
print(f"  Strong prompt: {strong_mean:.3f}  [{strong_lo:.3f}, {strong_hi:.3f}]")
print(f"  Weak prompt:   {weak_mean:.3f}  [{weak_lo:.3f}, {weak_hi:.3f}]")

# Check interval overlap
overlap = strong_lo < weak_hi and weak_lo < strong_hi
print(f"\n  Confidence intervals overlap: {overlap}")
print(f"  Conclusion: {'difference is NOT statistically distinguishable' if overlap else 'regression is statistically significant'}")

We visualize the two distributions to make the overlap (or separation) visually clear:

In [ ]:
#| code-fold: true
%config InlineBackend.figure_formats = ['svg']
import matplotlib.pyplot as plt

def _bootstrap_distribution(scores, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    arr = np.array(scores, dtype=float)
    return [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_bootstrap)]

strong_dist = _bootstrap_distribution(strong_scores)
weak_dist   = _bootstrap_distribution(weak_scores)

fig, ax = plt.subplots(figsize=(7, 3.5))

ax.hist(strong_dist, bins=30, alpha=0.65, color="#3a7ebf", label=f"Strong prompt (mean={strong_mean:.3f})")
ax.hist(weak_dist,   bins=30, alpha=0.65, color="#e05a4e", label=f"Weak prompt (mean={weak_mean:.3f})")

ax.axvline(strong_lo, color="#3a7ebf", linestyle="--", linewidth=1)
ax.axvline(strong_hi, color="#3a7ebf", linestyle="--", linewidth=1)
ax.axvline(weak_lo,   color="#e05a4e", linestyle="--", linewidth=1)
ax.axvline(weak_hi,   color="#e05a4e", linestyle="--", linewidth=1)

ax.set_xlabel("Bootstrap mean faithfulness")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap distributions of faithfulness (1000 resamples, n=20)")
ax.legend()
ax.grid(alpha=0.4, linestyle="dashed")
plt.tight_layout()
plt.show()

**Figure.** Bootstrap distributions of mean faithfulness for the strong and weak prompt pipelines over 1000 resamples of 20 examples. Dashed vertical lines mark the 95% confidence interval boundaries. Non-overlapping intervals indicate a statistically detectable regression; overlapping intervals mean the observed difference could plausibly be sampling noise.

We define a helper that formats a metric result in the reporting style recommended for eval notes — always report mean with CI, never a bare number:

In [ ]:
def report_metric(
    name: str,
    scores: list[float],
    n_bootstrap: int = 1000,
    confidence: float = 0.95,
) -> str:
    """Return a human-readable metric report string with confidence interval."""
    mean, lo, hi = bootstrap_ci(scores, n_bootstrap=n_bootstrap, confidence=confidence)
    pct = int(confidence * 100)
    return f"{name}: {mean:.3f}  (95% CI [{lo:.3f}, {hi:.3f}], n={len(scores)})"


print(report_metric("faithfulness (strong)", strong_scores))
print(report_metric("faithfulness (weak)",   weak_scores))

# Demonstrate: a small difference on a small set may not be meaningful
rng2 = np.random.default_rng(7)
run_a = rng2.binomial(1, 0.80, size=20).tolist()
run_b = rng2.binomial(1, 0.82, size=20).tolist()
print()
print("Runs with nominally different means (0.80 vs 0.82, n=20):")
print("  ", report_metric("run_a", run_a))
print("  ", report_metric("run_b", run_b))

_, lo_a, hi_a = bootstrap_ci(run_a)
_, lo_b, hi_b = bootstrap_ci(run_b)
overlap_ab = lo_a < hi_b and lo_b < hi_a
print(f"  Intervals overlap: {overlap_ab} — {('indistinguishable from noise' if overlap_ab else 'detectable difference')}")

The last block demonstrates the key point: a faithfulness change from 0.80 to 0.82 on a 20-example dataset almost always has overlapping bootstrap CIs — the difference is within sampling noise. This is why eval sets should be as large as practical. Doubling from 20 to 40 examples roughly halves the CI width (variance scales as $1/n$), making a 0.02 improvement detectable at 95% confidence.

:::{.callout-note}
For production eval sets with $n \geq 100$ examples, a 0.02 improvement in faithfulness is typically statistically significant at the 95% level. For the 20-example golden datasets we build in this series, only differences larger than $\approx 0.08$ should be treated as real. Calibrate your regression thresholds accordingly.

:::

## Exercises

1. **Extend `DatasetCurator` with multi-hop examples.** Add a `generate_multihop` method that selects two passages from different sections and asks the LLM to write a question whose complete answer requires information from both passages. These are harder for RAG pipelines and better stress-test the retrieval component.

2. **Add position-bias correction to the pairwise judge.** Extend `LLMJudge.pairwise` to run each comparison twice with A and B swapped, then return the majority verdict. If the judge disagrees between the two orderings, return `preferred="tie"` with `confidence="tie"`. Test this on a pair of nearly equivalent answers and verify the tie rate is higher than with a single-direction comparison.

3. **Implement metric trend detection.** Write a function `detect_trend(registry: PromptVersionRegistry, metric: str, window: int = 5)` that reads the last `window` entries from the registry and returns `"degrading"` if the metric shows a monotonically decreasing trend, `"improving"` if monotonically increasing, or `"stable"` otherwise. Populate the registry with 8 synthetic entries with a gradual faithfulness decay and verify the function returns `"degrading"`.

---

$\blacksquare$